# Lektion 12: Pandas

---

<div style="background-color: #fffde7; border: 2px solid #f9a825; border-radius: 6px; padding: 12px 18px;">
<h2 style="margin-top: 0;">Lernziele</h2>
<p>Nach dieser Lektion können Sie:</p>
<ul>
 <li>die Grenzen homogener <code>numpy</code>-Arrays bei gemischten Datentypen erklären,</li>
 <li><code>Series</code> und <code>DataFrame</code> aus eigenen Daten erzeugen und ihre Metadaten lesen,</li>
 <li>fehlende Werte als <code>NaN</code> erkennen und ihr Verhalten in Auswertungen erklären,</li>
 <li>Messdaten mit <code>pd.read_csv()</code> einlesen und die Argumente an das Dateiformat anpassen,</li>
 <li>Spalten, Zeilen und Werte über <code>[]</code>, <code>loc</code> und <code>iloc</code> auswählen sowie mit Masken filtern und sortieren.</li>
</ul>
</div>

**Inhalt**

 * [Motivation](#pandas)
 * [Einführung in `pandas`](#dataframes)
 * [Daten einlesen](#readcsv)
 * [Daten auswählen](#select)

---

## <a id=pandas></a> 1. Motivation und Abgrenzung zu `numpy`

In den bisherigen Lektionen stehen `numpy`-Arrays im Mittelpunkt. Für eine einzelne Messreihe genügt das; sobald eine Tabelle Spalten mit unterschiedlichen Datentypen enthält, stößt das Array an eine Grenze.

Ein typisches Beispiel sind drei Angaben zu derselben Messung: Messzeit, Messwert und Ort der Aufnahme.

In [1]:
import numpy as np

zeit = np.array([0.0, 1.0, 2.0, 3.0])
wert = np.array([2.1, 2.4, 2.2, 2.8])
ort = np.array(['Labor', 'Labor', 'Flur', 'Flur'])

print(zeit)
print(wert)
print(ort)

[0. 1. 2. 3.]
[2.1 2.4 2.2 2.8]
['Labor' 'Labor' 'Flur' 'Flur']


Die drei Arrays gehören zusammen: `zeit[2]`, `wert[2]` und `ort[2]` beschreiben dieselbe Messung. Diese Zuordnung steht nur im Kopf der Person, die den Code geschrieben hat; der Rechner kennt sie nicht.

### 1.1 Grenzen von `numpy`-Arrays

Ein gemeinsames 2d-Array scheint die Zuordnung zu sichern. Es setzt jedoch voraus, dass alle Elemente denselben Typ besitzen:

In [2]:
tabelle = np.array([zeit, wert, ort])

print(tabelle)
print(tabelle.dtype)

[['0.0' '1.0' '2.0' '3.0']
 ['2.1' '2.4' '2.2' '2.8']
 ['Labor' 'Labor' 'Flur' 'Flur']]
<U32


`numpy` wandelt die Zahlen in Text um, damit alle Elemente denselben Typ `<U32` erhalten. Mit diesem Array lässt sich nicht mehr rechnen; die Information über Messzeit und Messwert ist verloren.

### 1.2 Was `pandas` zusätzlich leistet

`pandas` ergänzt `numpy` um die fehlenden Möglichkeiten. Eine Tabelle trägt Spaltennamen und einen Zeilenindex, jede Spalte behält ihren eigenen Datentyp, fehlende Werte haben eine eigene Darstellung `NaN`. Filtern, Sortieren und Auswerten erfolgen über Namen statt über Spaltennummern.

Die numerischen Operationen im Hintergrund stammen weiterhin von `numpy`; `pandas` verwaltet die Beschriftungen und die Struktur. Beide Bibliotheken ergänzen sich daher, statt sich zu ersetzen.

---

## <a id=dataframes></a> 2. Einführung in `pandas`

`pandas` stellt zwei Container bereit: die *Series* für eine beschriftete Spalte und den *DataFrame* für eine vollständige Tabelle. Beide tragen einen Index, der die Zeilen bezeichnet.

### 2.1 Series

Eine `Series` entsteht aus einer Liste. Ohne weitere Angabe vergibt `pandas` einen `RangeIndex` mit den Zahlen `0` bis `n-1`, wie bei `numpy`-Arrays:

In [3]:
import pandas as pd

werte = pd.Series([2.1, 2.4, 2.2, 2.8])

print(werte)
print(werte.index)
print(werte.dtype)

0    2.1
1    2.4
2    2.2
3    2.8
dtype: float64
RangeIndex(start=0, stop=4, step=1)
float64


Der Index lässt sich selbst festlegen. Er muss nicht numerisch sein, darf aber keine doppelten Labels enthalten:

In [4]:
messung = pd.Series([2.1, 2.4, 2.2, 2.8],
                    index=['A', 'B', 'C', 'D'])

print(messung)
print(messung['C'])
print(messung[['A', 'C']])

A    2.1
B    2.4
C    2.2
D    2.8
dtype: float64
2.2
A    2.1
C    2.2
dtype: float64


Der Zugriff erfolgt über das Index-Label, nicht über die Position. `messung[['A', 'C']]` wählt zwei Einträge gleichzeitig und liefert wieder eine `Series`.

### 2.2 DataFrame

Ein *DataFrame* ist eine Sammlung von Spalten mit gemeinsamem Index. Üblicherweise erzeugen Sie ihn aus einem Dictionary, dessen Schlüssel die Spaltennamen und dessen Werte die Spalteninhalte sind:

In [5]:
daten = pd.DataFrame({
    'zeit': [0.0, 1.0, 2.0, 3.0],
    'wert': [2.1, 2.4, 2.2, 2.8],
    'ort': ['Labor', 'Labor', 'Flur', 'Flur'],
})

print(daten)

   zeit  wert    ort
0   0.0   2.1  Labor
1   1.0   2.4  Labor
2   2.0   2.2   Flur
3   3.0   2.8   Flur


Die Schlüssel des Dictionaries legen die Spaltennamen fest, die Listen die Werte. Alle Listen müssen dieselbe Länge besitzen; den Zeilenindex ergänzt `pandas` automatisch.

### 2.3 Metadaten und neue Spalten

Wie ein `numpy`-Array besitzt ein `DataFrame` Metadaten. Sie beschreiben die Form, die Beschriftungen und die Datentypen:

In [6]:
print(daten.shape)
print(daten.index)
print(daten.columns)
print(daten.dtypes)
print(type(daten))

(4, 3)
RangeIndex(start=0, stop=4, step=1)
Index(['zeit', 'wert', 'ort'], dtype='object')
zeit    float64
wert    float64
ort      object
dtype: object
<class 'pandas.core.frame.DataFrame'>


`shape` gibt die Zahl der Zeilen und Spalten zurück. `index` und `columns` enthalten die Beschriftungen, `dtypes` den Typ jeder Spalte. Text erscheint dort als `object`, weil `pandas` Zeichenketten nicht über einen eigenen `numpy`-Typ speichert.

Eine Spalte wählen Sie über ihren Namen; das Ergebnis ist eine `Series`. Eine neue Spalte entsteht durch Zuweisung an einen noch nicht vergebenen Namen:

In [7]:
print(daten['wert'])

daten['fehler'] = [0.1, 0.1, 0.2, 0.15]
daten['wert_quadrat'] = daten['wert'] ** 2

print(daten)

0    2.1
1    2.4
2    2.2
3    2.8
Name: wert, dtype: float64
   zeit  wert    ort  fehler  wert_quadrat
0   0.0   2.1  Labor    0.10          4.41
1   1.0   2.4  Labor    0.10          5.76
2   2.0   2.2   Flur    0.20          4.84
3   3.0   2.8   Flur    0.15          7.84


Die Rechnung auf der rechten Seite wirkt auf die gesamte Spalte, ohne Schleife. Das Ergebnis wird Zeile für Zeile dem gemeinsamen Index zugeordnet.

### 2.4 Fehlende Werte

Fehlende Messwerte sind in Tabellen der Normalfall. `pandas` kennzeichnet sie mit `NaN` (*Not a Number*) und lässt sie in Auswertungen zunächst aus:

In [8]:
daten.loc[2, 'wert'] = np.nan

print(daten)
print(daten['wert'].mean())
print(daten.isna())

   zeit  wert    ort  fehler  wert_quadrat
0   0.0   2.1  Labor    0.10          4.41
1   1.0   2.4  Labor    0.10          5.76
2   2.0   NaN   Flur    0.20          4.84
3   3.0   2.8   Flur    0.15          7.84
2.433333333333333
    zeit   wert    ort  fehler  wert_quadrat
0  False  False  False   False         False
1  False  False  False   False         False
2  False   True  False   False         False
3  False  False  False   False         False


Der Mittelwert berücksichtigt nur die drei vorhandenen Werte. Mit `skipna=False` liefert dieselbe Rechnung stattdessen `NaN`. `isna()` zeigt für jede Zelle, ob ein Wert fehlt.

---

## <a id=readcsv></a> 3. Daten einlesen

Messdaten liegen meist als Textdatei vor. Lektion 09 hat dafür `np.loadtxt()` verwendet; `pandas` liest dieselben Dateien mit `pd.read_csv()` und liefert direkt eine benannte Tabelle.

### 3.1 Das Beispiel aus Lektion 09

In [9]:
!cat data/data.txt

# data table
# x-data values
# y-data sqr values
1       1
2       4
3       9
4       16
5       25
10      100      
100     10000
1000    1000000


Die Datei enthält drei Kommentarzeilen und acht Messpunkte, deren Spalten durch Leerzeichen getrennt sind.

In [10]:
import numpy as np

data_np = np.loadtxt('data/data.txt')

print(data_np)
print(data_np.dtype)

[[1.0e+00 1.0e+00]
 [2.0e+00 4.0e+00]
 [3.0e+00 9.0e+00]
 [4.0e+00 1.6e+01]
 [5.0e+00 2.5e+01]
 [1.0e+01 1.0e+02]
 [1.0e+02 1.0e+04]
 [1.0e+03 1.0e+06]]
float64


`np.loadtxt()` liefert ein 2d-Array ohne Spaltennamen. Die Bedeutung der beiden Spalten ergibt sich allein aus ihrer Reihenfolge.

In [11]:
import pandas as pd

data = pd.read_csv('data/data.txt', comment='#', sep=r'\s+',
                   names=['x', 'y'])

print(data)

      x        y
0     1        1
1     2        4
2     3        9
3     4       16
4     5       25
5    10      100
6   100    10000
7  1000  1000000


`comment='#'` überspringt die Kommentarzeilen, `sep=r'\s+'` behandelt jede Folge von Leerzeichen als Trennzeichen. `names` vergibt die Spaltennamen; ohne diese Angabe würde `pandas` die erste Datenzeile als Kopfzeile verwenden.

In [12]:
print(data.shape)
print(data.columns)
print(data.dtypes)
print(data.index)

(8, 2)
Index(['x', 'y'], dtype='object')
x    int64
y    int64
dtype: object
RangeIndex(start=0, stop=8, step=1)


Die Spalten tragen jetzt Namen, die Werte bleiben Ganzzahlen (`int64`). Den Index hat `pandas` automatisch vergeben; er zählt die Zeilen von `0` bis `7`.

### 3.2 CSV-Dateien mit Kommentaren

Im zweiten Beispiel aus Lektion 09 beginnen die Kommentare mit `;`, getrennt wird mit Komma. Die Datei enthält keine Kopfzeile, dafür drei Datenspalten.

In [13]:
!cat data/data.csv

; use this as a comment
; 
; this is a Comma Separated Values file
; x-val, sqr, cubic
0, 0, 0
1, 1, 0
2, 4, 8
3, 9, 27
4, 16, 64

In [14]:
data_csv = pd.read_csv('data/data.csv', comment=';', sep=',',
                       header=None, names=['x', 'sqr', 'cubic'],
                       usecols=['x', 'cubic'])

print(data_csv)

   x  cubic
0  0      0
1  1      0
2  2      8
3  3     27
4  4     64


`header=None` verhindert, dass die erste Datenzeile als Kopfzeile gelesen wird. `names` legt die Namen aller drei Spalten fest, `usecols` wählt daraus die erste und die dritte aus. Das Ergebnis entspricht der Auswahl `usecols=(0, 2)` aus Lektion 09.

Die Argumente lassen sich einzeln anpassen; eine vollständige Übersicht bietet die Dokumentation von `pandas`.

---

## <a id=select></a> 4. Daten auswählen

`pandas` bietet mehrere Wege zur Auswahl. Sie unterscheiden sich darin, ob sie über Namen oder über Positionen arbeiten; wer den Unterschied kennt, vermeidet falsche Zeilen.

### 4.1 Spalten auswählen

Die eckigen Klammern mit einem Spaltennamen liefern eine `Series`. Eine Liste von Namen erzeugt einen neuen `DataFrame` mit genau diesen Spalten:

In [15]:
print(data['x'])
print(type(data['x']))
print(data[['y', 'x']])

0       1
1       2
2       3
3       4
4       5
5      10
6     100
7    1000
Name: x, dtype: int64
<class 'pandas.core.series.Series'>
         y     x
0        1     1
1        4     2
2        9     3
3       16     4
4       25     5
5      100    10
6    10000   100
7  1000000  1000


Die Reihenfolge in der Liste bestimmt die Reihenfolge der Spalten im Ergebnis. Die Zeilen bleiben vollständig erhalten; ihre Index-Labels ändern sich nicht.

### 4.2 `loc` und `iloc`

Für Zeilen gibt es zwei Zugriffswege. `loc` adressiert über das Index-Label, `iloc` über die Position in der Tabelle. Bei einem automatisch vergebenen Index stimmen beide zunächst überein, nach dem Sortieren nicht mehr.

In [16]:
print(data.loc[2])
print(data.loc[2, 'y'])

print(data.iloc[2])
print(data.iloc[2, 1])

x    3
y    9
Name: 2, dtype: int64
9
x    3
y    9
Name: 2, dtype: int64
9


`loc[2]` liefert die Zeile mit dem Label `2` als `Series`, `loc[2, 'y']` daraus einen einzelnen Wert. `iloc[2]` liefert die dritte Zeile, `iloc[2, 1]` deren zweiten Eintrag. Vor dem Komma steht die Zeilenauswahl, dahinter die Spaltenauswahl.

### 4.3 Filtern mit Masken

Ein Vergleich auf einer Spalte liefert eine `Series` mit Wahrheitswerten. Diese *Maske* wählt in eckigen Klammern genau die Zeilen aus, für die der Vergleich zutrifft:

In [17]:
print(data['y'] > 20)
print(data[data['y'] > 20])

0    False
1    False
2    False
3    False
4     True
5     True
6     True
7     True
Name: y, dtype: bool
      x        y
4     5       25
5    10      100
6   100    10000
7  1000  1000000


Das Ergebnis behält alle Spalten und die ursprünglichen Index-Labels. Mehrere Bedingungen werden mit `&` verknüpft und jeweils in Klammern gesetzt:

In [18]:
print(data[(data['x'] >= 2) & (data['y'] < 500)])

    x    y
1   2    4
2   3    9
3   4   16
4   5   25
5  10  100


`and` funktioniert an dieser Stelle nicht, weil es die gesamte `Series` als einen Wahrheitswert auswertet. Der Operator `&` arbeitet elementweise, wie die Vergleichsoperatoren selbst.

`loc` nimmt vor dem Komma die Zeilenauswahl und dahinter die Spaltenauswahl entgegen. Damit lässt sich eine gefilterte Tabelle auf einzelne Spalten reduzieren:

In [19]:
gross = data['y'] > 1000

print(data.loc[gross, 'x'])
print(data.loc[gross, ['x', 'y']])

6     100
7    1000
Name: x, dtype: int64
      x        y
6   100    10000
7  1000  1000000


Für Zuweisungen ist dieser Weg der sichere: `data.loc[gross, 'y'] = 1000` ändert nur die ausgewählten Zellen. Eine Kette aus zwei eckigen Klammern kann dagegen auf einer Kopie arbeiten, sodass die Änderung unbemerkt verloren geht.

### 4.4 Sortieren

`sort_values()` ordnet die Zeilen nach einer Spalte und gibt eine neue Tabelle zurück; die Ausgangstabelle bleibt unverändert:

In [20]:
print(data.sort_values('y'))
print(data.sort_values('y', ascending=False))

      x        y
0     1        1
1     2        4
2     3        9
3     4       16
4     5       25
5    10      100
6   100    10000
7  1000  1000000
      x        y
7  1000  1000000
6   100    10000
5    10      100
4     5       25
3     4       16
2     3        9
1     2        4
0     1        1


Mit `ascending=False` kehrt sich die Reihenfolge um. Die Index-Labels wandern mit ihren Zeilen und werden nicht neu vergeben. `sort_index()` stellt die ursprüngliche Reihenfolge wieder her.

### 4.5 Ausgewählte Daten auswerten

Eine ausgewählte Spalte ist eine `Series` und kennt deren Methoden. Damit lässt sich das Ergebnis einer Auswahl unmittelbar prüfen:

In [21]:
print(data['y'].mean())
print(data['y'].max())
print(data.loc[gross, 'y'].mean())

126269.375
1000000
505000.0


Der Mittelwert der Spalte `y` unterscheidet sich deutlich von dem der gefilterten Werte. Solche Gegenproben zeigen, ob eine Maske die erwarteten Zeilen auswählt.

---

<div style="background-color: #e8f5e9; border: 2px solid #1b5e20; border-radius: 6px; padding: 12px 18px;">
<h2 style="margin-top: 0;">Zusammenfassung</h2>
<ul>
 <li><code>pandas</code> ergänzt <code>numpy</code> um benannte Spalten, gemischte Datentypen und fehlende Werte (<code>NaN</code>).</li>
 <li>Eine <code>Series</code> ist eine beschriftete Spalte, ein <code>DataFrame</code> eine Tabelle mit Zeilenindex und Spaltennamen.</li>
 <li><code>pd.read_csv()</code> liest Textdateien; <code>sep</code>, <code>comment</code>, <code>header</code>, <code>names</code> und <code>usecols</code> passen das Einlesen an.</li>
 <li><code>loc</code> adressiert über Labels, <code>iloc</code> über Positionen; eckige Klammern mit einer Namensliste wählen Spalten.</li>
 <li>Boolesche Masken filtern Zeilen, <code>sort_values()</code> ordnet sie, ohne die Ausgangstabelle zu ändern.</li>
</ul>
</div>